# Phase 3 — MobileNetV2 training and evaluation
Training has NOT been executed. Choose a GPU runtime. This notebook embeds the repository training pipeline.

Use PlantVillage color images only (mohanty/PlantVillage card: CC-BY-SA-3.0). Preserve attribution/share-alike obligations. Do not mix grayscale/segmented versions. Prepare a CSV with path,label,leaf_group,split. Use official leaf-grouped test membership and split remaining training leaf groups for validation. Keep all exact and near duplicates in one split. The pipeline audits this and aborts on leakage. Do not tune on test results.

Store images and manifest in Google Drive. Paths in the manifest are relative to the manifest folder. If no official split is usable, partition leaf groups with seed 42 into 70/15/15 and record this decision.


In [ ]:
%pip install tensorflow==2.21.0 pandas scikit-learn matplotlib pillow ImageHash


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
Path("/content/train_disease.py").write_text('"""Reproducible MobileNetV2 training from a licensed, leaf-grouped manifest.\n\nCSV columns: path,label,leaf_group,split (train/validation/test).\nUse the official grouped test set; split training groups for validation.\nPaths resolve relative to the manifest. No datasets are bundled.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\ndef prepare(manifest):\n    import pandas as pd\n    import imagehash\n    from PIL import Image\n    frame = pd.read_csv(manifest)\n    required = {\'path\', \'label\', \'leaf_group\', \'split\'}\n    if not required.issubset(frame.columns) or frame[list(required)].isna().any().any():\n        raise ValueError(\'Manifest requires path,label,leaf_group,split without missing values\')\n    if set(frame.split) != {\'train\', \'validation\', \'test\'}:\n        raise ValueError(\'All three explicit splits required\')\n    if frame.groupby(\'leaf_group\').split.nunique().max() != 1:\n        raise ValueError(\'Leaf group leakage\')\n    seen, hashes, rows, rejected = {}, {}, [], []\n    for row in frame.to_dict(\'records\'):\n        path = (Path(manifest).resolve().parent / row[\'path\']).resolve()\n        try:\n            with Image.open(path) as im:\n                im.load()\n                digest = hashlib.sha256(im.convert(\'RGB\').tobytes()).hexdigest()\n                perceptual = imagehash.phash(im)\n        except (OSError, ValueError):\n            rejected.append(str(path))\n            continue\n        if digest in seen:\n            if seen[digest] != (row[\'split\'], row[\'label\']):\n                raise ValueError(\'Exact duplicate leakage or conflicting label\')\n            continue\n        # Exhaustive Hamming-distance audit, never silently overlooks cross-split neighbors.\n        for previous, split in hashes.values():\n            if split != row[\'split\'] and perceptual - previous <= 4:\n                raise ValueError(\'Near-duplicate across splits; regroup and rerun\')\n        seen[digest] = (row[\'split\'], row[\'label\'])\n        hashes[digest] = (perceptual, row[\'split\'])\n        rows.append({**row, \'path\': str(path), \'sha256\': digest})\n    clean = pd.DataFrame(rows)\n    labels = sorted(clean.label.unique())\n    for split in [\'train\', \'validation\', \'test\']:\n        if set(clean[clean.split == split].label) != set(labels):\n            raise ValueError(\'Every class must occur in every split\')\n    return clean, labels, rejected\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'manifest\')\n    parser.add_argument(\'--license\', required=True, help=\'Verified dataset license and attribution\')\n    parser.add_argument(\'--output\', default=\'backend/artifacts\')\n    parser.add_argument(\'--baseline\', help=\'Optional compatible original .keras model, same labels/preprocessing\')\n    args = parser.parse_args()\n    import numpy as np\n    import tensorflow as tf\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n    tf.keras.utils.set_random_seed(42)\n    tf.config.experimental.enable_op_determinism()\n    out = Path(args.output); out.mkdir(parents=True, exist_ok=True)\n    frame, labels, rejected = prepare(args.manifest)\n    frame.to_csv(out / \'disease-split.csv\', index=False)\n    (out / \'disease-data-audit.json\').write_text(json.dumps({\'counts\':frame.groupby([\'split\',\'label\']).size().reset_index(name=\'count\').to_dict(\'records\'), \'corrupt\': rejected},indent=2))\n    def dataset(split):\n        part = frame[frame.split == split]\n        ds = tf.data.Dataset.from_tensor_slices((part.path.tolist(), [labels.index(x) for x in part.label]))\n        def decode(path, label):\n            image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)\n            image.set_shape([None, None, 3])\n            return tf.image.resize(tf.cast(image, tf.float32), (224,224)) / 127.5 - 1, label\n        ds = ds.map(decode)\n        if split == \'train\': ds = ds.shuffle(len(part), seed=42)\n        return ds.batch(32).prefetch(tf.data.AUTOTUNE)\n    train, val, test = [dataset(s) for s in [\'train\',\'validation\',\'test\']]\n    base = tf.keras.applications.MobileNetV2(include_top=False, weights=\'imagenet\', input_shape=(224,224,3))\n    base.trainable = False\n    inputs = tf.keras.Input((224,224,3))\n    x = tf.keras.layers.RandomFlip(\'horizontal\')(inputs)\n    x = tf.keras.layers.RandomRotation(.08)(x)\n    x = base(x, training=False)\n    x = tf.keras.layers.GlobalAveragePooling2D()(x)\n    outputs = tf.keras.layers.Dense(len(labels), activation=\'softmax\')(x)\n    model = tf.keras.Model(inputs, outputs)\n    def fit(lr, epochs):\n        model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=\'sparse_categorical_crossentropy\', metrics=[\'accuracy\'])\n        return model.fit(train, validation_data=val, epochs=epochs, callbacks=[\n            tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),\n            tf.keras.callbacks.ModelCheckpoint(str(out/\'disease.keras\'), save_best_only=True)])\n    warm = fit(.001, 10)\n    warm_loss = min(warm.history[\'val_loss\'])\n    warm_path = out/\'disease-warm.keras\'; model.save(warm_path)\n    base.trainable = True\n    for layer in base.layers[:-20]: layer.trainable = False\n    fine = fit(.00001, 20)\n    if min(fine.history[\'val_loss\']) >= warm_loss:\n        model = tf.keras.models.load_model(warm_path)\n    else:\n        model = tf.keras.models.load_model(out/\'disease.keras\')\n    model.save(out/\'disease.keras\')\n    truth = np.concatenate([y.numpy() for _, y in test])\n    def evaluate(candidate):\n        predicted = candidate.predict(test).argmax(axis=1)\n        return {\'accuracy\': accuracy_score(truth,predicted), \'report\': classification_report(truth,predicted,labels=list(range(len(labels))),target_names=labels,output_dict=True,zero_division=0), \'confusionMatrix\': confusion_matrix(truth,predicted,labels=list(range(len(labels)))).tolist()}\n    metrics = {\'fineTuned\': evaluate(model), \'original\': None, \'fieldEvaluation\': \'pending\'}\n    plt.figure(figsize=(16,14))\n    plt.imshow(metrics[\'fineTuned\'][\'confusionMatrix\'],cmap=\'Blues\')\n    plt.xticks(range(len(labels)),labels,rotation=90,fontsize=6)\n    plt.yticks(range(len(labels)),labels,fontsize=6)\n    plt.xlabel(\'Predicted\');plt.ylabel(\'Actual\');plt.tight_layout()\n    plt.savefig(out/\'disease-confusion-matrix.png\');plt.close()\n    if args.baseline:\n        original = tf.keras.models.load_model(args.baseline, compile=False)\n        if original.output_shape[-1] != len(labels): raise ValueError(\'Baseline labels incompatible\')\n        metrics[\'original\'] = evaluate(original)\n    for metric in [\'accuracy\',\'loss\']:\n        plt.figure()\n        for key in [metric,\'val_\'+metric]: plt.plot(warm.history[key]+fine.history[key],label=key)\n        plt.legend(); plt.savefig(out/f\'disease-{metric}.png\'); plt.close()\n    (out/\'disease-evaluation.json\').write_text(json.dumps(metrics,indent=2))\n    meta = {\'file\':\'disease.keras\',\'version\':\'mobilenetv2-finetuned-seed42\',\'labels\':labels,\'preprocessing\':\'rgb_mobilenet_v2\',\n            \'license\':args.license,\'sha256\':hashlib.sha256((out/\'disease.keras\').read_bytes()).hexdigest(),\n            \'metrics\':metrics,\'configuration\':{\'seed\':42,\'batch\':32,\'warmEpochs\':len(warm.epoch),\'fineEpochs\':len(fine.epoch),\'manifestSha256\':hashlib.sha256(Path(args.manifest).read_bytes()).hexdigest()}}\n    (out/\'disease.json\').write_text(json.dumps(meta,indent=2))\n\nif __name__ == \'__main__\': main()\n')


In [ ]:
MANIFEST = "/content/drive/MyDrive/PlantVillage/manifest.csv"
OUTPUT = "/content/drive/MyDrive/AgriSmart-disease-artifacts"
assert Path(MANIFEST).is_file(), "Prepare the licensed, leaf-grouped manifest first"


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"/content/train_disease.py",MANIFEST,"--license","CC-BY-SA-3.0; PlantVillage, Mohanty et al. 2016","--output",OUTPUT],check=True)


In [ ]:
import json
from IPython.display import display, Image
print(Path(OUTPUT,"disease-evaluation.json").read_text())
for metric in ["accuracy","loss"]: display(Image(filename=str(Path(OUTPUT,f"disease-{metric}.png"))))


## Comparison and deployment
The runtime initial model uses Transformers/PyTorch with its bundled resize/crop processor. The fine-tuned artifact here uses Keras and RGB MobileNetV2 normalization. A same-dataset comparison must use each model's documented processor and identical held-out image IDs. Do not pass the Transformers safetensors file to --baseline (that option is for compatible Keras artifacts only). Independent field evaluation remains pending until appropriately licensed labeled photographs are available. Copy disease.keras and disease.json into backend/artifacts after inspecting the report.
